# 📖 Notebook 5: SQL JOINs Deep Dive

JOINs are one of the most important concepts in SQL. They let you **combine rows from two or more tables** based on a related column. This notebook is a complete visual guide — every JOIN type, with diagrams, queries, and results.

## Learning Objectives

- Understand and use all 6 SQL JOIN types: INNER, LEFT, RIGHT, FULL OUTER, CROSS, SELF
- Visualize what each JOIN includes and excludes
- Chain multiple JOINs together
- Combine JOINs with aggregation (COUNT, SUM, GROUP BY)
- Avoid common JOIN mistakes

## 🛠️ Setup

Start the database first:

```bash
cd 01-foundations/data-modeling
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "data_modeling_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    """Run a SELECT and return rows as dictionaries."""
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    """Run an INSERT/UPDATE/DELETE and commit."""
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.commit()
    conn.close()

def show(rows):
    """Pretty-print query results as a table."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(str(c).ljust(widths[c]) for c in cols)
    print(header)
    print("-+-".join("-" * widths[c] for c in cols))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(widths[c]) for c in cols))

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

## 🏗️ Create Demo Tables

We'll create two small tables that make it easy to see exactly what each JOIN does.

- **departments** — 5 departments (some have no employees)
- **employees** — 8 employees (some have no department)

```
departments                    employees
┌────┬──────────────┐         ┌────┬──────────┬─────────┬───────────┐
│ id │ name         │         │ id │ name     │ dept_id │ manager_id│
├────┼──────────────┤         ├────┼──────────┼─────────┼───────────┤
│  1 │ Engineering  │    ┌───▶│  1 │ Alice    │    1    │   NULL    │
│  2 │ Marketing    │    │    │  2 │ Bob      │    1    │     1     │
│  3 │ Sales        │    │    │  3 │ Charlie  │    2    │   NULL    │
│  4 │ HR           │    │    │  4 │ Diana    │    3    │   NULL    │
│  5 │ Research     │◄──no──┐ │  5 │ Eve      │    1    │     1     │
└────┴──────────────┘  emp  │ │  6 │ Frank    │   NULL  │     3     │
                             │ │  7 │ Grace    │   NULL  │     4     │
                             │ │  8 │ Hank     │    4    │     3     │
                             │ └────┴──────────┴─────────┴───────────┘
                             └── Frank & Grace have no department
                                 Research (5) has no employees
```

This setup gives us:
- Rows that match (employees with departments)
- Left-only rows (employees with `NULL` dept_id)
- Right-only rows (departments with no employees)
- Self-referencing (manager_id → employees.id)

In [ ]:
conn = get_db()
cursor = conn.cursor()

cursor.execute("""
    DROP TABLE IF EXISTS employees CASCADE;
    DROP TABLE IF EXISTS departments CASCADE;
    DROP TABLE IF EXISTS projects CASCADE;

    CREATE TABLE departments (
        id   SERIAL PRIMARY KEY,
        name VARCHAR(50) NOT NULL
    );

    CREATE TABLE employees (
        id         SERIAL PRIMARY KEY,
        name       VARCHAR(50) NOT NULL,
        dept_id    INTEGER REFERENCES departments(id),
        manager_id INTEGER REFERENCES employees(id),
        salary     INTEGER DEFAULT 60000
    );

    CREATE TABLE projects (
        id      SERIAL PRIMARY KEY,
        name    VARCHAR(50) NOT NULL,
        dept_id INTEGER REFERENCES departments(id),
        budget  INTEGER DEFAULT 100000
    );

    INSERT INTO departments (id, name) VALUES
        (1, 'Engineering'),
        (2, 'Marketing'),
        (3, 'Sales'),
        (4, 'HR'),
        (5, 'Research');

    -- Alice and Eve report to nobody (top-level)
    -- Bob and Eve are in Engineering under Alice
    -- Frank and Grace have NO department (dept_id = NULL)
    INSERT INTO employees (id, name, dept_id, manager_id, salary) VALUES
        (1, 'Alice',   1, NULL, 120000),
        (2, 'Bob',     1, 1,    95000),
        (3, 'Charlie', 2, NULL, 85000),
        (4, 'Diana',   3, NULL, 90000),
        (5, 'Eve',     1, 1,    110000),
        (6, 'Frank',   NULL, 3, 70000),
        (7, 'Grace',   NULL, 4, 65000),
        (8, 'Hank',    4, 3,    75000);

    INSERT INTO projects (id, name, dept_id, budget) VALUES
        (1, 'Website Redesign', 2, 50000),
        (2, 'Mobile App',       1, 200000),
        (3, 'Sales Dashboard',  3, 80000),
        (4, 'AI Platform',      1, 300000),
        (5, 'Hiring Portal',    4, 40000);

    SELECT setval('departments_id_seq', 5);
    SELECT setval('employees_id_seq', 8);
    SELECT setval('projects_id_seq', 5);
""")
conn.commit()
conn.close()

print("✅ Tables created: departments, employees, projects")
print("\n--- departments ---")
show(query("SELECT * FROM departments ORDER BY id"))
print("\n--- employees ---")
show(query("SELECT id, name, dept_id, manager_id, salary FROM employees ORDER BY id"))

---

## 1️⃣ INNER JOIN — Only Matching Rows

An **INNER JOIN** returns only rows where there's a match in **both** tables.

```
  employees          departments
 ┌─────────┐       ┌─────────┐
 │         │       │         │
 │   ┌─────┼───────┼─────┐   │
 │   │█████│ INNER │█████│   │
 │   │█████│ JOIN  │█████│   │
 │   └─────┼───────┼─────┘   │
 │  Frank  │       │Research │
 │  Grace  │       │(no emp) │
 └─────────┘       └─────────┘
  excluded            excluded
```

**Frank** and **Grace** (no dept_id) are excluded.  
**Research** (no employees) is excluded.

In [ ]:
rows = query("""
    SELECT e.name AS employee, d.name AS department
    FROM employees e
    INNER JOIN departments d ON e.dept_id = d.id
    ORDER BY e.id
""")

show(rows)
print(f"\n→ {len(rows)} rows. Frank, Grace (no dept) and Research (no employees) are excluded.")

**Expected result:**

| employee | department  |
|----------|-------------|
| Alice    | Engineering |
| Bob      | Engineering |
| Charlie  | Marketing   |
| Diana    | Sales       |
| Eve      | Engineering |
| Hank     | HR          |

6 rows — only employees **with** a department, and only departments **with** employees.

---

## 2️⃣ LEFT JOIN — All From Left, Matching From Right

A **LEFT JOIN** returns **all rows from the left table** (employees), plus matching rows from the right table (departments). If there's no match, the right side fills with `NULL`.

```
  employees          departments
 ┌─────────┐       ┌─────────┐
 │█████████│       │         │
 │█████┌───┼───────┼───┐█████│
 │█████│███│ LEFT  │███│     │
 │█████│███│ JOIN  │███│     │
 │█████└───┼───────┼───┘     │
 │█Frank██ │       │Research │
 │█Grace██ │       │(NULL)   │
 └─────────┘       └─────────┘
  ALL included       excluded
```

Frank and Grace appear with `NULL` department.  
Research is still excluded (no employee references it).

In [ ]:
rows = query("""
    SELECT e.name AS employee, d.name AS department
    FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.id
    ORDER BY e.id
""")

show(rows)
print(f"\n→ {len(rows)} rows. Frank and Grace show NULL department.")

**Expected result:**

| employee | department  |
|----------|-------------|
| Alice    | Engineering |
| Bob      | Engineering |
| Charlie  | Marketing   |
| Diana    | Sales       |
| Eve      | Engineering |
| Frank    | None        |
| Grace    | None        |
| Hank     | HR          |

8 rows — **all** employees, even those without a department.

### 💡 Finding rows with NO match

A common pattern: "Find all employees who are NOT in any department."

In [ ]:
# LEFT JOIN + WHERE IS NULL = find unmatched rows
rows = query("""
    SELECT e.name AS employee
    FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.id
    WHERE d.id IS NULL
    ORDER BY e.name
""")

show(rows)
print("\n→ These employees have no department assigned.")

---

## 3️⃣ RIGHT JOIN — All From Right, Matching From Left

A **RIGHT JOIN** is the mirror of LEFT JOIN: **all rows from the right table** (departments), plus matching from the left.

```
  employees          departments
 ┌─────────┐       ┌─────────┐
 │         │       │█████████│
 │     ┌───┼───────┼───┐█████│
 │     │███│ RIGHT │███│█████│
 │     │███│ JOIN  │███│█████│
 │     └───┼───────┼───┘█████│
 │  Frank  │       │Research█│
 │  Grace  │       │(NULL←)██│
 └─────────┘       └─────────┘
  excluded           ALL included
```

In [ ]:
rows = query("""
    SELECT e.name AS employee, d.name AS department
    FROM employees e
    RIGHT JOIN departments d ON e.dept_id = d.id
    ORDER BY d.id, e.id
""")

show(rows)
print(f"\n→ {len(rows)} rows. Research appears with NULL employee. Frank/Grace excluded.")

**Expected result:**

| employee | department  |
|----------|-------------|
| Alice    | Engineering |
| Bob      | Engineering |
| Eve      | Engineering |
| Charlie  | Marketing   |
| Diana    | Sales       |
| Hank     | HR          |
| None     | Research    |

7 rows — **all** departments, even Research which has no employees.

💡 **Tip**: You can always rewrite a RIGHT JOIN as a LEFT JOIN by swapping the table order. Most developers prefer LEFT JOIN for consistency.

---

## 4️⃣ FULL OUTER JOIN — Everything From Both

A **FULL OUTER JOIN** returns **all rows from both tables**. Where there's no match, the missing side fills with `NULL`.

```
  employees          departments
 ┌─────────┐       ┌─────────┐
 │█████████│       │█████████│
 │█████┌───┼───────┼───┐█████│
 │█████│███│ FULL  │███│█████│
 │█████│███│ OUTER │███│█████│
 │█████└───┼───────┼───┘█████│
 │█Frank██ │       │Research█│
 │█Grace██ │       │█████████│
 └─────────┘       └─────────┘
  ALL included       ALL included
```

In [ ]:
rows = query("""
    SELECT e.name AS employee, d.name AS department
    FROM employees e
    FULL OUTER JOIN departments d ON e.dept_id = d.id
    ORDER BY COALESCE(e.id, 99), COALESCE(d.id, 99)
""")

show(rows)
print(f"\n→ {len(rows)} rows. Both unmatched employees AND unmatched departments appear.")

**Expected result:**

| employee | department  |
|----------|-------------|
| Alice    | Engineering |
| Bob      | Engineering |
| Charlie  | Marketing   |
| Diana    | Sales       |
| Eve      | Engineering |
| Frank    | None        |
| Grace    | None        |
| Hank     | HR          |
| None     | Research    |

9 rows — **everybody** from both sides.

---

## 5️⃣ CROSS JOIN — Every Combination

A **CROSS JOIN** produces the **Cartesian product** — every row from the left paired with every row from the right. No `ON` condition needed.

```
3 employees × 2 departments = 6 rows

  Alice  ──┬── Engineering
  Alice  ──┴── Marketing
  Bob    ──┬── Engineering
  Bob    ──┴── Marketing
  Charlie──┬── Engineering
  Charlie──┴── Marketing
```

⚠️ **Warning**: 1000 rows × 1000 rows = **1,000,000** result rows! Use with small tables only.

In [ ]:
# Cross join a subset to keep output manageable
rows = query("""
    SELECT e.name AS employee, d.name AS department
    FROM (SELECT * FROM employees WHERE id <= 3) e
    CROSS JOIN (SELECT * FROM departments WHERE id <= 2) d
    ORDER BY e.id, d.id
""")

show(rows)
print(f"\n→ {len(rows)} rows (3 employees × 2 departments). Every possible combination.")

### 💡 When is CROSS JOIN useful?

- **Generate all time slots**: `dates × hours`
- **Generate test combinations**: `sizes × colors`
- **Comparison matrices**: every product vs every product

---

## 6️⃣ SELF JOIN — Table Joined With Itself

A **SELF JOIN** is when a table is joined to **itself**. This is useful for hierarchical data like employee → manager relationships.

```
employees (as e)          employees (as m)
┌────┬─────────┬────┐     ┌────┬─────────┐
│ id │ name    │mgr │────▶│ id │ name    │
├────┼─────────┼────┤     ├────┼─────────┤
│  2 │ Bob     │  1 │────▶│  1 │ Alice   │  Bob's manager is Alice
│  5 │ Eve     │  1 │────▶│  1 │ Alice   │  Eve's manager is Alice
│  6 │ Frank   │  3 │────▶│  3 │ Charlie │  Frank's manager is Charlie
│  8 │ Hank    │  3 │────▶│  3 │ Charlie │  Hank's manager is Charlie
│  7 │ Grace   │  4 │────▶│  4 │ Diana   │  Grace's manager is Diana
└────┴─────────┴────┘     └────┴─────────┘
```

In [ ]:
# Self join: find each employee's manager
rows = query("""
    SELECT
        e.name  AS employee,
        m.name  AS manager
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.id
    ORDER BY e.id
""")

show(rows)
print("\n→ LEFT JOIN so employees without managers (Alice, Charlie, Diana) show NULL.")

In [ ]:
# Find who manages the most people
rows = query("""
    SELECT
        m.name       AS manager,
        COUNT(e.id)  AS direct_reports
    FROM employees e
    INNER JOIN employees m ON e.manager_id = m.id
    GROUP BY m.id, m.name
    ORDER BY direct_reports DESC
""")

show(rows)
print("\n→ Alice manages Bob and Eve. Charlie manages Frank and Hank.")

---

## 7️⃣ Multiple JOINs — Chaining 3+ Tables

In real applications, you often join 3 or more tables. Just chain the JOIN clauses:

```
employees ──JOIN──▶ departments ──JOIN──▶ projects
   "who"              "which dept"         "which project"
```

In [ ]:
# Three-way join: employee → department → projects in that department
rows = query("""
    SELECT
        e.name  AS employee,
        d.name  AS department,
        p.name  AS project,
        p.budget
    FROM employees e
    INNER JOIN departments d ON e.dept_id = d.id
    INNER JOIN projects p    ON p.dept_id = d.id
    ORDER BY d.name, e.name, p.name
""")

show(rows)
print(f"\n→ {len(rows)} rows. Each employee paired with every project in their department.")

In [ ]:
# Mix JOIN types: all employees, their dept, and any projects (if they exist)
rows = query("""
    SELECT
        e.name  AS employee,
        d.name  AS department,
        p.name  AS project
    FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.id
    LEFT JOIN projects p    ON p.dept_id = d.id
    ORDER BY e.name, p.name
""")

show(rows)
print(f"\n→ {len(rows)} rows. Frank and Grace show NULL dept AND NULL project.")

---

## 8️⃣ JOINs with Aggregation — COUNT, SUM, GROUP BY

JOINs become really powerful when combined with aggregation.

In [ ]:
# Count employees per department (including departments with 0 employees)
rows = query("""
    SELECT
        d.name          AS department,
        COUNT(e.id)     AS employee_count
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.id
    GROUP BY d.id, d.name
    ORDER BY employee_count DESC
""")

show(rows)
print("\n→ LEFT JOIN ensures Research appears with 0. COUNT(e.id) ignores NULLs.")

In [ ]:
# Total salary budget per department
rows = query("""
    SELECT
        d.name                 AS department,
        COUNT(e.id)            AS headcount,
        COALESCE(SUM(e.salary), 0) AS total_salary,
        COALESCE(AVG(e.salary)::int, 0) AS avg_salary
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.id
    GROUP BY d.id, d.name
    ORDER BY total_salary DESC
""")

show(rows)

In [ ]:
# Departments with total project budget
rows = query("""
    SELECT
        d.name                    AS department,
        COUNT(DISTINCT e.id)      AS employees,
        COUNT(DISTINCT p.id)      AS projects,
        COALESCE(SUM(DISTINCT p.budget), 0) AS total_budget
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.id
    LEFT JOIN projects p  ON p.dept_id = d.id
    GROUP BY d.id, d.name
    ORDER BY total_budget DESC
""")

show(rows)
print("\n→ DISTINCT prevents double-counting when multiple JOINs produce row multiplication.")

---

## ⚠️ Common JOIN Mistakes

### Mistake 1: Forgetting the ON clause (accidental CROSS JOIN)

```sql
-- ❌ BAD: This is a cross join! 8 × 5 = 40 rows
SELECT e.name, d.name
FROM employees e, departments d;

-- ✅ GOOD: Always use explicit JOIN with ON
SELECT e.name, d.name
FROM employees e
INNER JOIN departments d ON e.dept_id = d.id;
```

### Mistake 2: Ambiguous column names

```sql
-- ❌ BAD: 'name' exists in both tables
SELECT name FROM employees e JOIN departments d ON e.dept_id = d.id;

-- ✅ GOOD: Always qualify with table alias
SELECT e.name AS employee, d.name AS department
FROM employees e JOIN departments d ON e.dept_id = d.id;
```

### Mistake 3: NULL surprises with JOINs

```sql
-- ❌ WRONG: COUNT(*) counts NULL rows too!
SELECT d.name, COUNT(*) FROM departments d
LEFT JOIN employees e ON e.dept_id = d.id
GROUP BY d.name;
-- Research shows 1 (it counts the NULL row!)

-- ✅ RIGHT: COUNT(e.id) ignores NULLs
SELECT d.name, COUNT(e.id) FROM departments d
LEFT JOIN employees e ON e.dept_id = d.id
GROUP BY d.name;
-- Research shows 0 ✅
```

In [ ]:
# Demonstrate the COUNT(*) vs COUNT(column) trap
print("❌ COUNT(*) — counts NULL rows:")
show(query("""
    SELECT d.name AS department, COUNT(*) AS wrong_count
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.id
    GROUP BY d.name ORDER BY d.name
"""))

print("\n✅ COUNT(e.id) — ignores NULLs:")
show(query("""
    SELECT d.name AS department, COUNT(e.id) AS correct_count
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.id
    GROUP BY d.name ORDER BY d.name
"""))

print("\n→ Notice Research: COUNT(*) says 1, COUNT(e.id) correctly says 0.")

### Mistake 4: Row multiplication with multiple JOINs

When you JOIN to two tables that both have multiple matching rows, you get a **Cartesian explosion**:

In [ ]:
# Engineering has 3 employees and 2 projects → 3 × 2 = 6 rows for Engineering
rows = query("""
    SELECT d.name AS dept, e.name AS employee, p.name AS project
    FROM departments d
    JOIN employees e ON e.dept_id = d.id
    JOIN projects p  ON p.dept_id = d.id
    WHERE d.name = 'Engineering'
    ORDER BY e.name, p.name
""")

show(rows)
print(f"\n→ {len(rows)} rows! 3 employees × 2 projects. Fix with DISTINCT or subqueries.")

---

## 🧹 Clean Up

In [ ]:
# The tables persist in the Docker container.
# To remove them:
# execute("DROP TABLE IF EXISTS projects, employees, departments CASCADE")

print("✅ Tables left in place for further experimentation.")
print("   To reset: docker compose down -v && docker compose up -d")

---

## 📚 Summary — JOIN Cheat Sheet

| JOIN Type | Returns | NULL Behavior |
|-----------|---------|---------------|
| **INNER JOIN** | Only matching rows from both tables | No NULLs |
| **LEFT JOIN** | All left rows + matching right rows | Right side NULL if no match |
| **RIGHT JOIN** | Matching left rows + all right rows | Left side NULL if no match |
| **FULL OUTER JOIN** | All rows from both tables | NULLs on both sides where no match |
| **CROSS JOIN** | Every left row × every right row | No NULLs (Cartesian product) |
| **SELF JOIN** | Table joined with itself | Depends on JOIN type used |

### Key Takeaways

1. **INNER JOIN** is the most common — use it when you only want matching rows
2. **LEFT JOIN** is essential for "find all X, even those without Y" queries
3. **LEFT JOIN + WHERE IS NULL** is the pattern for finding unmatched rows
4. **RIGHT JOIN** is rare — just swap the table order and use LEFT JOIN
5. **FULL OUTER JOIN** is for reconciliation — "show me everything from both sides"
6. **CROSS JOIN** produces huge results — use only with small tables
7. **SELF JOIN** is for hierarchies (employee → manager, category → parent)
8. Always use **table aliases** and **qualify column names** to avoid ambiguity
9. Use **COUNT(column)** not **COUNT(\*)** with LEFT JOIN to avoid counting NULLs
10. Watch for **row multiplication** when joining multiple one-to-many relationships